In [1]:
# 1. Installing the packages

!pip install requests beautifulsoup4 markdownify openai ftfy python-dotenv

In [2]:
# 2. Importing the libraries

import os
import re
import requests

from bs4 import BeautifulSoup, Comment
from ftfy import fix_text
from markdownify import markdownify as markdownify_html
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [3]:
# 3. Loading the environment variable

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

client = OpenAI(api_key=api_key)

In [5]:
# 4. Verify the environment setup

print("API key loaded:", bool(api_key)) 

API key loaded: True


In [6]:
MODEL_NAME = "gpt-5.4-nano"

## 5. Fetching the Webpage

In [7]:
def fetch_page(url: str) -> str:
    """
    Download the HTML content from a webpage.
    """
    headers = {
        "User-Agent": "SimpleAIScraper/1.0"
    }

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    return response.text

In [8]:
raw = fetch_page("https://www.olostep.com/")
print(raw[:500])

<!DOCTYPE html><!--  This site was created in Webflow. https://webflow.com  --><!--  Last Published: Tue Jul 28 2026 13:08:45 GMT+0000 (Coordinated Universal Time)  -->
<html data-wf-page="69f2ff131a05ff32f831b78b" data-wf-site="6665ee948e0a3f19d93e90c7">
<head>
  <meta charset="utf-8">
  <title>Olostep | Web Data API for Search, Scraping &amp; Crawling</title>
  <meta content="Olostep powers AI with clean web data through one API for search, scraping, and crawling. Get structured data extractio


## 6. Cleaning the HTML
 

In [9]:
def clean_html(html):
    html = fix_text(html)

    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious noisy tags
    for tag in soup([
        "script", "style", "noscript", "svg", "img", "iframe",
        "nav", "header", "footer", "aside", "form", "button"
    ]):
        tag.decompose()

    noise_words = [
        "cursor",
        "modal",
        "popup",
        "floating",
        "signup",
        "login",
        "cookie",
        "banner",
        "navbar",
        "menu",
        "footer",
        "header",
        "subscribe",
        "newsletter",
        "loading",
        "wait",
        "success",
        "auth",
        "w-nav",
        "w-form"
    ]

    # First collect noisy tags
    tags_to_remove = []

    for tag in soup.find_all(True):
        if tag.attrs is None:
            continue

        class_value = tag.get("class", [])
        id_value = tag.get("id", "")

        if isinstance(class_value, list):
            class_text = " ".join(class_value).lower()
        else:
            class_text = str(class_value).lower()

        id_text = str(id_value).lower()

        if any(word in class_text or word in id_text for word in noise_words):
            tags_to_remove.append(tag)

    # Then remove them safely
    for tag in tags_to_remove:
        tag.decompose()

    body = soup.body if soup.body else soup

    return str(body)

In [10]:
clean = clean_html(raw)
print(clean[:500])

<body>


<div class="page-wrapper-6 sticky-on-page">
<div class="preloader"></div>
<div class="global-styles-6 w-embed">

</div>

<main class="main-wrapper-4 more" id="main-section">
<section class="hero-section-pages">
<div class="container-hero-pages">
<div class="text-button-center">
<div class="w-layout-vflex flex-block-12">
<h1 class="h1">Web Data Infrastructure for AI</h1>
</div>
<div class="w-layout-vflex sub-flex-block-12">
<p class="sub-heading">Built to power the Web's second user, Olo


# 7. Converting HTML to Markdown
 

In [11]:
def html_to_markdown(html):
    markdown_text = markdownify_html(
        html,
        heading_style="ATX",
        bullets="-"
    )

    markdown_text = fix_text(markdown_text)

    # Remove image markdown
    markdown_text = re.sub(r"!\[.*?\]\(.*?\)", "", markdown_text)

    # Remove extra spaces and blank lines
    markdown_text = re.sub(r"[ \t]+", " ", markdown_text)
    markdown_text = re.sub(r"\n{3,}", "\n\n", markdown_text)

    lines = []

    skip_lines = [
        "click to try",
        "wait...",
        "you've successfully reserved your spot.",
        "thank you! your submission has been received!",
        "oops! something went wrong while submitting the form.",
        "product",
        "resources",
        "company"
    ]

    for line in markdown_text.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.lower() in skip_lines:
            continue

        lines.append(line)

    return "\n".join(lines)

In [12]:
md = html_to_markdown(clean)
print(md[:500])

# Web Data Infrastructure for AI
Built to power the Web's second user, Olostep is the best web search, scraping and crawling API for AI
[Start for free](/auth)
[Contact Sales](https://www.olostep.com/contact-sales)
## Trusted by the best startups **startups** in the world
## One **API** to Automate Web Data
Search, scrape, structure and monitor the whole web with one
API. Reliable, cost-effective, scalable. Handling Billions of requests
[### Monitor
Set up monitors for events happening across th


## 8.Asking a User Query Against the Page
 

In [13]:
def answer_query_from_page(markdown_text, user_query):
    prompt = f"""
You are an AI web scraping assistant.

You will receive Markdown extracted from a webpage.

Your task is to answer the user's query using only the useful page content.

User query:
{user_query}

Webpage Markdown:
{markdown_text}

Instructions:
- Return only clean Markdown.
- Use only information from the webpage Markdown.
- Do not invent missing details.
- Ignore navigation links, buttons, CTAs, popups, decorative labels, image captions, and repeated marketing fragments.
- Ignore lines like "Start for free", "Contact Sales", "Your AI Agent", and decorative workflow examples unless they directly answer the query.
- Focus on headings, paragraphs, product descriptions, feature sections, pricing details, documentation text, and factual claims.
- If the page does not contain the answer, say: "The page does not contain this information."
- Keep the answer short, clear, and focused.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )

    return response.output_text

## 9.Creating the Full AI Web Scraper
 

In [14]:
def ai_web_scraper(url, user_query):
    raw_html = fetch_page(url)
    cleaned_html = clean_html(raw_html)
    markdown_text = html_to_markdown(cleaned_html)
    answer = answer_query_from_page(markdown_text, user_query)

    return answer

## 10. Testing the AI Web Scraper
 

In [15]:
url = "https://www.olostep.com/"
user_query = "What does this company do?"
result = ai_web_scraper(url, user_query)

display(Markdown(result))

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}